In [1]:
%pip install scikeras

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from scikeras.wrappers import KerasClassifier

from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, BatchNormalization, GlobalAveragePooling1D
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

In [3]:
train = "final.csv"
test = "test.csv"

df = pd.read_csv(train)
test_df = pd.read_csv(test)

X = df.drop(columns=["label_tactic"]).values
y = df["label_tactic"].values

X_test = test_df.drop(columns=["label_tactic"]).values
y_test = test_df["label_tactic"].values

In [4]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(np.unique(y_encoded))

print("Number of classes:", num_classes)

Number of classes: 8


In [5]:
X = X.reshape(X.shape[0], X.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)

Train shape: (120000, 88, 1)
Test shape: (18582, 88, 1)


In [6]:
early_stop = EarlyStopping(
    monitor="loss",
    patience=5,
    restore_best_weights=True
)

In [7]:
def build_cnn(filters=64, lr=0.001):

    model = Sequential([
        Conv1D(filters, 3, padding="same", activation="relu", input_shape=(X.shape[1],1)),
        BatchNormalization(),
        MaxPooling1D(2),

        Conv1D(filters*2, 3, padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling1D(2),

        Conv1D(filters*4, 3, padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling1D(2),

        GlobalAveragePooling1D(),

        Dense(128, activation="relu"),
        Dropout(0.5),

        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [8]:
cnn_model = KerasClassifier(
    model=build_cnn,
    verbose=0
)

In [9]:
param_grid = {
    "epochs": [100, 200],
    "model__filters": [64, 128],
    "model__lr": [0.01, 0.1]
}

In [10]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [11]:
grid_search = GridSearchCV(
    cnn_model,
    param_grid,
    cv=skf,
    scoring="f1_weighted",
    n_jobs=2,
    verbose=1
)

grid_search.fit(X, y_encoded, callbacks=[early_stop])

print("\nBest parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Fitting 10 folds for each of 8 candidates, totalling 80 fits


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
best_params = grid_search.best_params_

final_model = build_cnn(
    filters=best_params["model__filters"],
    lr=best_params["model__lr"]
)

history = final_model.fit(
    X,
    y_encoded,
    epochs=best_params["epochs"],
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
predictions = final_model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)

accuracy = accuracy_score(y_test_encoded, predicted_classes)
f1 = f1_score(y_test_encoded, predicted_classes, average="weighted")

print("\nTest Accuracy:", accuracy)
print("Test F1 Score:", f1)

In [ ]:
print("\nClassification Report:")

print(
    classification_report(
        y_test_encoded,
        predicted_classes,
        target_names=label_encoder.classes_
    )
)

cm = confusion_matrix(y_test_encoded, predicted_classes)

print("\nFalse Positive Rate (per class)")

fpr_list = []

for i in range(len(cm)):
    FP = cm[:, i].sum() - cm[i, i]
    TN = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    
    fpr = FP / (FP + TN) if (FP + TN) != 0 else 0
    fpr_list.append(fpr)
    
    print(f"{label_encoder.classes_[i]}: {fpr:.4f}")

print("\nAverage FPR:", np.mean(fpr_list))

In [ ]:
y_test_pred = np.argmax(final_model.predict(X_test), axis=1)
cm = confusion_matrix(y_test_encoded, y_test_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

plt.figure(figsize=(12, 12))
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix - Test Set")
plt.show()